# Lezione 7 · Fondamenti delle macchine a vettori di supporto (SVM)

En las clases cinco y seis descubriste la clasificación con K vecinos más cercanos (KNN, del inglés K-Nearest Neighbors): un algoritmo intuitivo que decide la categoría de un dato nuevo mirando a sus vecinos. Funciona bien y es fácil de explicar, por eso suele ser el primer modelo que pruebas en un proyecto de clasificación.

Pero KNN tiene límites. Necesita guardar todos los datos de entrenamiento, se vuelve lento con conjuntos grandes y es sensible al ruido cuando el valor de K no encaja con el problema. Hoy abres un bloque nuevo dedicado a otra familia de algoritmos de clasificación: las máquinas de vectores de soporte (SVM, del inglés Support Vector Machine).

Una SVM ataca el mismo problema —separar datos en categorías— desde un ángulo distinto: en lugar de votar entre vecinos, busca la frontera que meglio separa las clases y la mantiene lo más alejada posible de los datos de cada lado. ¿Suena simple? Lo es, y precisamente por eso se convierte en una de las herramientas más robustas de la inteligencia artificial (IA) aplicada, con usos en diagnóstico médico, reconocimiento de imágenes, bioinformática y clasificación de texto.

Hoy te centras en la intuición geométrica de la SVM: qué es un vector, qué es un hiperplano separador y por qué maximizar el margen entre clases es la clave del algoritmo. También la aplicarás con scikit-learn sobre el mismo dataset médico de la clase cinco, para comparar directamente sus resultados con los de KNN. Guarda la calculadora de fórmulas complejas en el cajón: hoy toca entender el dibujo antes que la ecuación completa.

**Hai bisogno di Python installato insieme alle librerie scikit-learn y Matplotlib.** Se lavori in locale con Anaconda o con pip, installale così (in Google Colab sono già preinstallate, quindi puoi saltare questa cella):

```bash
pip install scikit-learn matplotlib
```

Programamos ogni blocco insieme, passo dopo passo. Apri questo notebook en Google Colab o nel tuo ambiente locale ed esegui cada celda man mano che avanziamo.

## Qué es un vector

Antes de hablar de SVM, necesitas dominar su ingrediente básico: el vector. Un vector es una magnitud que tiene, a la vez, longitud y dirección. Se suele representar como una flecha: el punto donde nace indica el origen, y la punta indica hacia dónde apunta y cuánto se extiende.

Piensa en una instrucción de senderismo: "camina tres pasos al norte". Esa frase describe un vector completo, con dirección ("norte") y longitud ("tres pasos"). Si en cambio caminas cinco pasos al este, obtienes otro vector distinto, con otra dirección y otra longitud. Dos vectores solo son iguales cuando coinciden en ambas cosas a la vez.

En matemáticas, un vector se representa como una lista de números, uno por cada dimensión del espacio en el que trabajas. En un plano de dos dimensiones, un vector `a` con valores 3 y 4 se escribe como (3, 4): avanza tres unidades en un eje y cuatro en el otro. ¿Ves ya hacia dónde va esto? En una SVM, cada dato de tu conjunto —cada tumor, cada correo, cada imagen— se convierte en un vector: cada característica medida ocupa una dimensión, y el conjunto de esos valores define un punto en un espacio con tantas dimensiones como caratteristiche tengas.

No todos los vectores representan datos. Más adelante conocerás el vector normal, que no describe ninguna muestra concreta, sino la orientación del propio hiperplano separador: apunta en la dirección perpendicular a la frontera, como una brújula clavada en mitad de la calle que marca hacia dónde mira. Con ese matiz en mente, ya distingues los dos papeles que cumplen los vectores dentro de una SVM: unos representan datos, y otro define la frontera que los separa.

### La magnitud de un vector

La magnitud de un vector es su longitud, y se calcula igual que la distancia euclídea que ya usaste con KNN: elevas al quadrato cada componente, sumas los cuadrados y sacas la raíz cuadrada del resultado. Para el vector (3, 4), la magnitud es la raíz cuadrada de (3² + 4²), es decir, la raíz cuadrada de 25, que da 5.

Fíjate en que este cálculo es una aplicación directa del teorema de Pitágoras: el vector actúa como la hipotenusa de un triángulo rectángulo cuyos catetos son sus componentes.

In [ ]:
from math import sqrt

vector_a = (3, 4)

magnitud = sqrt(vector_a[0] ** 2 + vector_a[1] ** 2)

print("Magnitud del vector a:", magnitud)

### Il prodotto scalare tra due vettori

Oltre ad avere una propria magnitudine, due vettori possono combinarsi tramite il prodotto scalare (detto anche prodotto punto): moltiplichi ogni componente di un vettore per la componente corrispondente dell'altro e somma i risultati. Se il vettore `a` vale (1, 3) e il vettore `b` vale (2, 4), il prodotto scalare è (1×2) + (3×4), che dà 10.

**Tieni a mente questa operazione**, perché è il pezzo che permette a una SVM di decidere su quale lato del confine cade un nuovo dato, come vedrai nella sezione seguente.

In [ ]:
vector_a = (1, 3)
vector_b = (2, 4)

producto_punto = vector_a[0] * vector_b[0] + vector_a[1] * vector_b[1]

print("Producto punto entre a y b:", producto_punto)

## La idea de SVM: separar clases con la calle más ancha posible

Imagina un mapa con dos barrios que quieres separar mediante una calle recta. Podrías trazar esa calle de muchas formas distintas, siempre que quede entre ambos barrios. Pero hay una opción meglio que las demás: la calle más ancha posible, la que deja el mayor espacio libre a ambos lados antes de tocar la primera vivienda de cada barrio.

Esa es exactamente la lógica de una SVM. La línea central de la calle es el hiperplano separador: la frontera que divide el espacio en dos regiones, una por clase. En un plano de dos dimensiones, el hiperplano es una línea recta; en tres dimensiones, es un plano; y en dimensiones superiores —como las 30 caratteristiche del dataset médico que usas hoy—, sigue llamándose hiperplano aunque ya no puedas dibujarlo.

Las viviendas más próximas a los bordes de la calle, una a cada lado, son las que definen su anchura. En una SVM, esos datos reciben el nombre de vectores de soporte: son los puntos de entrenamiento más cercanos a la frontera, y son los únicos que davvero importan para calcular dónde va el hiperplano. El resto de las viviendas, más alejadas hacia el interior de cada barrio, podría moverse sin afectar el trazado de la calle.

El ancho completo de la calle, medido de acera a acera, es el margen. El objetivo de una SVM es encontrar el hiperplano que maximiza ese margen. ¿Por qué te conviene una calle ancha? Cuanto más ancha sea, más espacio de seguridad tienes antes de equivocarte al clasificar un dato nuevo, y meglio generaliza el modelo ante datos que non ha visto durante el entrenamiento. Un margen estrecho, en cambio, deja el modelo pegado a los datos de entrenamiento y lo hace más frágil ante el ruido.

Piensa de nuevo en la calle entre los dos barrios. Si la trazas muy pegada a las viviendas de un lado, cualquier vivienda nueva que se construya un poco más adentro del otro barrio queda mal clasificada, porque la frontera no dejó margen de maniobra. Si en cambio queda bien centrada y ancha, tienes espacio de sobra para admitir pequeñas variaciones sin que la frontera falle. Con datos reales pasa lo mismo: un margen amplio tolera meglio el ruido y los valores atípicos, porque la frontera no depende de un puñado de casos extremos.

Antes de seguir, conviene que sepas por qué esta familia de algoritmos goza de tan buena reputación. Entre sus puntos fuertes destacan estos:

- Rinde bien en espacios con muchas caratteristiche, incluso cuando el número de caratteristiche supera al número de muestras
- El propio criterio del margen actúa como una forma de control contra el sobreajuste, ya que no se deja arrastrar por el ruido de los datos de entrenamiento
- Solo necesita guardar los vectores de soporte para clasificar datos nuevos, así que es más eficiente en memoria que un algoritmo que deba recordar todo el conjunto de entrenamiento, como hace KNN

Una vez trazada la calle, clasificar un dato nuevo es sencillo: solo necesitas mirar en qué acera cae. Matemáticamente, el hiperplano se describe con la ecuación:

```
w · x + b = 0
```

Aquí, `x` es el vector que representa el dato, `w` es el vector normal al hiperplano (perpendicular a la calle, marcando su orientación) y `b` es el término de sesgo, que desplaza la calle hacia un lado u otro sin cambiar su ángulo. Para clasificar un punto nuevo, calculas el producto punto entre `w` y `x`, le sumas `b`, y observas el signo del resultado: si es positivo o cero, el punto cae en un lado de la calle; si es negativo, cae en el otro.

Veámoslo con un esempio sencillo e inventado, solo para fijar la idea, con un vector `w` igual a (1, 1) y un sesgo `b` igual a -4:

In [ ]:
def clasificar(punto, w, b):
    producto = punto[0] * w[0] + punto[1] * w[1] + b
    if producto >= 0:
        return "positiva", producto
    return "negativa", producto


w = (1, 1)
b = -4

for punto in [(3, 3), (1, 1)]:
    clase, valor = clasificar(punto, w, b)
    print(f"Punto {punto} -> valor {valor} -> clase {clase}")

El punto (3, 3) da un resultado positivo y cae en un lado de la frontera; el punto (1, 1) da un resultado negativo y cae en el otro. Así funciona, en esencia, la previsione de cualquier SVM una vez entrenada: el trabajo pesado ocurre antes, cuando el algoritmo encuentra los valores óptimos de `w` y `b`.

## Por qué encontrar ese hiperplano es un problema de optimización

Trazar la calle más ancha suena sencillo cuando la dibujas a mano sobre un mapa con pocos puntos, pero encontrarla de forma exacta, con decenas de caratteristiche y miles de datos, exige matemáticas. Una SVM convierte esa búsqueda en un problema de optimización: encontrar los valores de `w` y `b` que maximizan el margen sin dejar ningún dato de entrenamiento mal clasificado.

Aquí aparece un resultado útil: el ancho del margen es inversamente proporcional a la magnitud del vector `w`. Cuanto menor sea esa magnitud, más ancha es la calle. Maximizar el margen equivale, entonces, a minimizar la magnitud de `w`, siempre respetando la condición de que cada dato de entrenamiento quede correctamente clasificado y fuera del margen.

Este tipo de problema —minimizar una función sujeta a restricciones— se conoce como programación cuadrática, y pertenece a una categoría especial llamada optimización convexa. La palabra "convexa" describe la forma del terreno de búsqueda: imagina un cuenco. Si sueltas una bola en cualquier punto de su superficie, siempre rueda hacia el mismo fondo, sin quedarse atascada en huecos intermedios. Eso es justo lo que ocurre con el problema de la SVM: por su naturaleza convexa, existe un único mínimo global, así que cualquier algoritmo que "ruede cuenco abajo" con cuidado termina encontrando la meglio solución posible.

Resolver este problema con precisión implica herramientas como los multiplicadores de Lagrange y técnicas de optimización cuadrática, que quedan fuera del alcance de esta clase. En la clase ocho programarás tú misma o tu stesso el algoritmo de entrenamiento passo dopo passo, y en la clase nueve profundizarás en el margen blando —que permite algunas clasificaciones erróneas mediante un parámetro llamado `C`— y en los kernels, que permiten separar datos que no son separables con una línea recta. Por ahora, quédate con la idea central: entrenar una SVM consiste en resolver un problema de optimización convexa que garantiza encontrar la meglio calle posible, no una calle cualquiera.

En la práctica, calcular la solución exacta con miles de datos sería demasiado lento si se resolviera de forma directa, así que scikit-learn recurre por dentro a algoritmos especializados, como la optimización mínima secuencial, que dividen el problema grande en muchos problemas pequeños y los resuelven uno a uno hasta converger. Tú no necesitas programar esa parte para usar `SVC`: la biblioteca ya la incorpora.

## Aplicar SVM con scikit-learn

Aunque la matemática completa espera a clases futuras, ya puedes entrenar y evaluar una SVM hoy mismo, grazie a la clase `SVC` (Support Vector Classification) de scikit-learn. Vas a repetir el mismo experimento de la clase cinco —clasificar tumores como benignos o malignos— para comparar directamente el resultado de SVM con el que obtuviste entonces con KNN.

### Paso uno: cargar los datos y dividirlos

Reutilizas el dataset médico ya incluido en scikit-learn, con la misma división entre entrenamiento y prueba que usaste con KNN, para que la comparación sea justa:

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

datos = load_breast_cancer()
X = datos.data
y = datos.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Muestras de entrenamiento:", len(X_train))
print("Muestras de prueba:", len(X_test))

Usar el mismo `random_state` que en la clase cinco asegura que ambos algoritmos ven exactamente la misma partición de datos, así que cualquier diferencia en la precisión final proviene del algoritmo, no del reparto aleatorio.

### Paso dos: crear, entrenar y evaluar la SVM

La clase `SVC` funciona con la misma lógica de tres pasos que ya conoces: creas el clasificador, lo entrenas con `fit()` y lo evalúas con `score()`. Empiezas con un kernel lineal, la opción más simple, que busca directamente una calle recta sin transformar el espacio de caratteristiche:

In [ ]:
from sklearn.svm import SVC

# Creas la SVM con un kernel lineal (la calle más simple: una línea recta)
clasificador_svm = SVC(kernel="linear")

# Entrenas el modelo: aquí ocurre la optimización convexa que viste antes
clasificador_svm.fit(X_train, y_train)

# Evalúas la precisión sobre los datos de prueba
precision_svm = clasificador_svm.score(X_test, y_test)
print("Precisión de la SVM:", precision_svm)

### Confronto rapido con KNN

E come si confronta questo risultato con quello ottenuto nella lezione cinque? Addestra un `KNeighborsClassifier` sulla stessa partizione di dati e confronta direttamente entrambe le precisioni:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

clasificador_knn = KNeighborsClassifier(n_neighbors=5)
clasificador_knn.fit(X_train, y_train)
precision_knn = clasificador_knn.score(X_test, y_test)

print("Precisión de la SVM:", precision_svm)
print("Precisión de KNN:", precision_knn)

En muchos problemas tabulares como este, la SVM iguala o incluso supera el resultado de KNN, aunque el valor exacto depende de la partición de datos, del kernel elegido y de otros parámetros. Lo importante ahora no es memorizar una cifra, sino comprobar con tus propios datos que dos algoritmos diversi, entrenados sobre la misma información, pueden llegar a decisiones parecidas por caminos completamente diferentes.

Ten presente, además, que la SVM comparte una particularidad con KNN: ambos algoritmos calculan distancias y productos punto entre vectores, así que ambos son sensibles a la escala de las caratteristiche. Si una columna se mide en miles y otra en unidades, la de números más grandes domina el resultado. Por simplicidad, este esempio entrena directamente sobre los datos originales, pero en un proyecto real conviene aplicar el mismo tipo de escalado que ya conoces, por esempio con `StandardScaler`, antes de entrenar la SVM.

### Paso tres: predecir un caso nuevo

Igual que con KNN, `predict()` te permite clasificar una muestra concreta:

In [ ]:
muestra_nueva = X_test[0].reshape(1, -1)

prediccion = clasificador_svm.predict(muestra_nueva)
clase_predicha = datos.target_names[prediccion[0]]

print("Clase predicha por la SVM:", clase_predicha)
print("Clase real:", datos.target_names[y_test[0]])

Si SVM y KNN coinciden en su previsione para este caso, ganas confianza en el diagnóstico; si difieren, tienes una señal clara de que ese caso concreto merece una revisión adicional antes de tomar cualquier decisión automática, algo especialmente relevante cuando el resultado alimenta un proceso clínico real.

## Visualiza la idea en dos dimensiones

Toda la teoría de la calle y el margen resulta más fácil de creer cuando la ves dibujada. El dataset médico tiene 30 caratteristiche, imposibles de representar en un gráfico, así que te quedas solo con dos de ellas —el radio medio y la textura media del tumor— para entrenar una SVM auxiliar y dibujar su frontera:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Nos quedamos solo con dos características, para poder dibujarlas en un plano
X_2d = X[:, :2]

X_2d_train, X_2d_test, y_2d_train, y_2d_test = train_test_split(
    X_2d, y, test_size=0.2, random_state=42
)

svm_2d = SVC(kernel="linear")
svm_2d.fit(X_2d_train, y_2d_train)

# Creamos una malla de puntos que cubre el rango de las dos características
x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200),
)

# Predecimos la clase para cada punto de la malla, para colorear cada región
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.2, cmap="coolwarm")
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap="coolwarm", edgecolors="k")
plt.xlabel(datos.feature_names[0])
plt.ylabel(datos.feature_names[1])
plt.title("Frontera de una SVM lineal con dos características")
plt.show()

Mira bien el gráfico antes de seguir leyendo. ¿Distingues la zona donde el color cambia de tono? Esa transición es una aproximación visual de la calle que separa las dos clases, con el hiperplano —aquí sí una línea, porque solo usas dos dimensiones— justo en el centro. Ten en cuenta que esta SVM auxiliar usa solo dos de las 30 caratteristiche, así que su precisión es menor que la del modelo completo que entrenaste antes: es una versión simplificada, pensada únicamente para que veas la idea con tus propios ojos.

## Un primer vistazo al parámetro C

Si revisas la documentación de `SVC`, verás que admite un parámetro llamado `C`, que por defecto vale 1.0. Sin entrar todavía en su significado matemático —eso llega en la clase nueve, cuando trabajes el margen blando—, puedes observar que cambiarlo también cambia el resultado:

In [ ]:
# Solo un vistazo: por ahora no profundices en qué representa C matemáticamente
for c in [0.01, 100]:
    modelo_c = SVC(kernel="linear", C=c)
    modelo_c.fit(X_train, y_train)
    precision_c = modelo_c.score(X_test, y_test)
    print(f"C = {c} -> precisión = {precision_c:.4f}")

Guarda esta observación para il notebook de práctica de esta clase, donde exploras el efecto de `C` con más valores y sobre un dataset distinto.

## Riassunto: lo que te llevas de esta clase

- Un vector es una magnitud con dirección y longitud, como la instrucción "camina tres pasos al norte"; en SVM, cada dato de entrenamiento se representa como un vector en un espacio de caratteristiche
- La magnitud de un vector se calcula con la misma fórmula que la distancia euclídea, y el producto punto combina dos vectores en un único número
- Una SVM busca el hiperplano —la "calle"— que separa las clases con el margen más ancho posible; los datos más cercanos a los bordes de esa calle son los vectores de soporte
- Encontrar ese hiperplano óptimo es un problema de optimización convexa: minimizar la magnitud del vector `w` sujeto a clasificar bien todos los datos, un problema con un único mínimo global
- Con scikit-learn entrenas una SVM en pocas líneas: `SVC`, `fit()` para entrenar, `predict()` para predecir y `score()` para medir la precisión, igual que con `KNeighborsClassifier`
- El parámetro `C` de `SVC` también influye en el resultado, pero su significado matemático llega en la clase nueve

## Mettilo in pratica

Ya viste cómo entrenar una SVM lineal y cómo un primer cambio en `C` afecta a la precisión. **¡Continúa ahora con il notebook de práctica de esta clase!** Allí aplicas el mismo procedura sobre un dataset distinto, con ejercicios guiados por TODOs.

## Sigle usate en esta clase

- **IA**: Inteligencia Artificial
- **KNN**: K-Nearest Neighbors o K vecinos más cercanos
- **SVM**: Support Vector Machine o máquinas de vectores de soporte